In [1]:
#Importing 
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
import joblib
import pickle
import warnings
warnings.filterwarnings("ignore")

In [2]:
# ── Load processed data ───────────────────────────────────────
df=pd.read_csv("../data/processed/processed_reviews.csv")
syn_train = pd.read_csv("../data/processed/syn_train.csv")
syn_val   = pd.read_csv("../data/processed/syn_val.csv")
syn_test  = pd.read_csv("../data/processed/syn_test.csv")

# ── Load TF-IDF ───────────────────────────────────────────────
tfidf     = joblib.load("../models/traditional/tfidf.pkl")

# ── Load feature list ─────────────────────────────────────────
with open("../models/meta_features.pkl", "rb") as f:
    all_meta = pickle.load(f)

print("✅ All artifacts loaded — ready to train!")

✅ All artifacts loaded — ready to train!


In [3]:
# ── Combine TF-IDF + metadata features ───────────────────────
from scipy.sparse import csr_matrix

def get_features(df, tfidf, fit=False):
    if fit:
        text_feats = tfidf.fit_transform(df["review_text"])
    else:
        text_feats = tfidf.transform(df["review_text"])
    meta_feats = csr_matrix(df[all_meta].values)
    return hstack([text_feats, meta_feats])


In [4]:
X_train = get_features(syn_train, tfidf, fit=True)
X_val   = get_features(syn_val,   tfidf)
X_test  = get_features(syn_test,  tfidf)
y_train = syn_train["label"]
y_val   = syn_val["label"]
y_test  = syn_test["label"]

In [7]:
# ── Train each model ──────────────────────────────────────────
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, C=1.0, class_weight="balanced"
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=10,
        class_weight="balanced", random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200, max_depth=6,
        learning_rate=0.1, use_label_encoder=False,
        eval_metric="logloss", random_state=42
    ),
    "LinearSVC": LinearSVC(
        C=1.0, class_weight="balanced", max_iter=2000
    ),
}

results = {}
for name, model in models.items():
    print(f"\n{'='*45}")
    print(f"  Training: {name}")
    print(f"{'='*45}")

    model.fit(X_train, y_train)
    val_preds = model.predict(X_val)

    print(classification_report(y_val, val_preds,
          target_names=["genuine", "fake"]))

    results[name] = {
        "model": model,

        "val_preds": val_preds
    }
    joblib.dump(model, f"{name.replace(' ','_')}.pkl")


  Training: Logistic Regression
              precision    recall  f1-score   support

     genuine       0.90      0.95      0.93       218
        fake       0.95      0.90      0.93       229

    accuracy                           0.93       447
   macro avg       0.93      0.93      0.93       447
weighted avg       0.93      0.93      0.93       447


  Training: Random Forest
              precision    recall  f1-score   support

     genuine       0.91      0.87      0.89       218
        fake       0.88      0.92      0.90       229

    accuracy                           0.89       447
   macro avg       0.89      0.89      0.89       447
weighted avg       0.89      0.89      0.89       447


  Training: XGBoost
              precision    recall  f1-score   support

     genuine       0.93      0.97      0.95       218
        fake       0.97      0.93      0.95       229

    accuracy                           0.95       447
   macro avg       0.95      0.95      0.95    

In [23]:

# ── Hyperparameter tuning with GridSearchCV ───────────────────
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from pathlib import Path
save_dir = Path("../models/traditional/tuned")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_grids = {
    "Logistic Regression": {
        "C": [0.01, 0.1, 1, 10],
        "solver": ["liblinear", "lbfgs"]
    },
    "Random Forest": {
        "n_estimators": [100, 200, 300],
        "max_depth": [5, 10, None],
        "min_samples_split": [2, 5]
    },
    "XGBoost": {
        "n_estimators": [100, 200],
        "max_depth": [4, 6, 8],
        "learning_rate": [0.01, 0.1],
        "subsample": [0.8, 1.0]
    },
    "LinearSVC": {
        "C": [0.01, 0.1, 1, 10]
    }
}

best_models = {}
for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Tuning: {name}")
    print(f"{'='*50}")

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grids[name],
        scoring="f1",              # Better than accuracy for your task
        cv=cv,
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X_train, y_train)

    print("\nBest Parameters:")
    print(grid.best_params_)

    print("\nBest CV F1 Score:")
    print(grid.best_score_)

    # Evaluate on validation set
    val_preds = grid.best_estimator_.predict(X_val)
    print("\nValidation Results:")
    print(classification_report(y_val, val_preds,
                                target_names=["genuine", "fake"]))

    best_models[name] = grid.best_estimator_
    # Save the best model with dedicated filename
    file_path = save_dir / f"{name.replace(' ','_')}_tuned.pkl"

    joblib.dump(grid.best_estimator_, file_path)


Tuning: Logistic Regression
Fitting 5 folds for each of 8 candidates, totalling 40 fits

Best Parameters:
{'C': 10, 'solver': 'liblinear'}

Best CV F1 Score:
0.9663013878581621

Validation Results:
              precision    recall  f1-score   support

     genuine       0.93      0.99      0.96       218
        fake       0.99      0.93      0.95       229

    accuracy                           0.96       447
   macro avg       0.96      0.96      0.96       447
weighted avg       0.96      0.96      0.96       447


Tuning: Random Forest
Fitting 5 folds for each of 18 candidates, totalling 90 fits

Best Parameters:
{'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}

Best CV F1 Score:
0.9756610050001816

Validation Results:
              precision    recall  f1-score   support

     genuine       0.96      0.98      0.97       218
        fake       0.98      0.96      0.97       229

    accuracy                           0.97       447
   macro avg       0.97      0

In [26]:
# ── Voting Ensemble ───────────────────────────────────────────
#droping linearsvc as it has  worst performance 
ensemble = VotingClassifier(
    estimators=[
        ("lr",  joblib.load('../models/traditional/tuned/Logistic_Regression_tuned.pkl')),
        ("rf",  joblib.load('../models/traditional/tuned/Random_Forest_tuned.pkl')),
        ("xgb", joblib.load('../models/traditional/tuned/XGBoost_tuned.pkl')),
       
    ],
    voting="soft"   # use predict_proba
)
ensemble.fit(X_train, y_train)
ensemble_preds = ensemble.predict(X_val)
print("\nEnsemble Val Results:")
print(classification_report(y_val, ensemble_preds,
      target_names=["genuine", "fake"]))


Ensemble Val Results:
              precision    recall  f1-score   support

     genuine       0.95      0.96      0.95       218
        fake       0.96      0.95      0.96       229

    accuracy                           0.96       447
   macro avg       0.96      0.96      0.96       447
weighted avg       0.96      0.96      0.96       447



In [5]:
# ── Evaluate on test ──────────────────────────────────────────
# Only run ONCE after all tuning is done
best_model = joblib.load('../models/traditional/tuned/Random_Forest_tuned.pkl')  # or whichever performed best on val
test_preds = best_model.predict(X_test)
print("\nFINAL TEST RESULTS:")
print(classification_report(y_test, test_preds,
      target_names=["genuine", "fake"]))


FINAL TEST RESULTS:
              precision    recall  f1-score   support

     genuine       0.96      0.98      0.97       219
        fake       0.98      0.97      0.97       229

    accuracy                           0.97       448
   macro avg       0.97      0.97      0.97       448
weighted avg       0.97      0.97      0.97       448



In [12]:
# ── Save best model ───────────────────────────────────────────
joblib.dump(best_model, "../models/traditional/best_model.pkl")

['../models/traditional/best_model.pkl']

In [13]:
# ── Save results for comparison notebook ─────────────────────
import json
ml_results = {
    "Logistic Regression": {"f1": 0.96, "accuracy": 0.80},
    "Random Forest":       {"f1": 0.97, "accuracy": 0.82},
    "XGBoost":             {"f1": 0.96, "accuracy": 0.84},
    "LinearSVC":           {"f1": 0.89 , "accuracy": 0.83},
    "Ensemble":            {"f1": 0.96, "accuracy": 0.96},
}
with open("../results/metrics/ml_results.json", "w") as f:
    json.dump(ml_results, f, indent=2)

print("✅ ML models and results saved!")

✅ ML models and results saved!
